*This will be the main notebook for practicing git*

# Git Solo Workflow — Reference Notes

---

## The Three Zones

Every file lives in one of three zones. Knowing which zone tells you what command to use.

**Zone 1 — Working Directory**
Files on disk. Git can see changes here but is not tracking them yet.

↓ `git add <file>`

**Zone 2 — Staging Area (Index)**
A holding area where you choose exactly what goes into the next commit.
You can stage some files and leave others out.

↓ `git commit -m "message"`

**Zone 3 — Repository (.git folder)**
Permanent history. Each commit gets a unique hash ID you can refer back to.

> The staging area exists so you can edit many files but commit only the ones that are ready.
> Clean, focused commits are a professional habit.

---

## Core Commands

| Command | What it does |
|---|---|
| `git init` | Create a new repository in the current folder |
| `git status` | Show which zone each changed file is in |
| `git add <file>` | Move a file from working directory → staging |
| `git commit -m "msg"` | Save staged changes as a permanent snapshot |
| `git log --oneline` | Show commit history, one line per commit |
| `git diff <file>` | Show line-by-line changes vs the last commit |

---

## Undoing Things

The right undo command depends on which zone the change is in.

### Zone 1 — Change is on disk, not yet staged
```
git restore <file>
```
⚠ **Destructive.** The change is gone permanently — git never knew about it.

### Zone 2 — Change is staged, not yet committed
```
git restore --staged <file>
```
✓ **Safe.** The change moves back to the working directory, nothing is lost.

### Zone 3 — Change is already committed

**Safe option — `git revert`**
```
git revert <hash>
```
Adds a new commit that cancels the target commit. History is preserved.
Use this on any shared or remote branch.

**Nuclear option — `git reset`**
```
git reset --soft <hash>   # uncommit, keep changes staged
git reset --mixed <hash>  # uncommit, unstage, keep files on disk (default)
git reset --hard <hash>   # uncommit and delete changes from disk
```
⚠ `--hard` is irreversible. Never use reset on a branch others are working on.

---

## Commit Message Convention

Format: `<type>: <short description>`

A good message completes: *"If applied, this commit will..."*

| Type | When to use |
|---|---|
| `feat` | New feature |
| `fix` | Bug fix |
| `docs` | Documentation only |
| `refactor` | Restructuring without behaviour change |
| `chore` | Housekeeping — config, dependencies, gitignore |

---

## `git status` and `git log` — Your Navigation Tools

> These two commands should be instinctive. Run them constantly. Neither changes anything — always safe.

**`git status`**
Shows where you are right now — what is modified, staged, or untracked.
Also prints hints about what to run next. Read the full output, not just the filenames.

**`git log --oneline`**
Shows where you have been — full commit history with short hashes.
Those hashes are how you refer to commits in `git diff`, `git revert`, and `git reset`.


In [2]:
from IPython.display import HTML, display

display(HTML("""
<div style="display:flex;flex-direction:column;align-items:center;font-family:system-ui,sans-serif;">
  <div style="display:flex;align-items:baseline;gap:16px;margin-bottom:10px;">
    <span style="font-size:14px;color:#888;">Score</span>
    <span id="score" style="font-size:22px;font-weight:600;">0</span>
    <span style="font-size:14px;color:#888;margin-left:12px;">Best</span>
    <span id="hi" style="font-size:16px;font-weight:600;color:#888;">0</span>
  </div>
  <canvas id="c" style="border:1px solid #ccc;border-radius:8px;" tabindex="1"></canvas>
  <p id="msg" style="margin:10px 0 2px;font-weight:600;font-size:16px;"></p>
  <p id="sub" style="margin:0;font-size:13px;color:#888;"></p>
  <div style="display:grid;grid-template-columns:repeat(3,44px);grid-template-rows:repeat(3,44px);gap:4px;margin-top:12px;">
    <div></div>
    <button onclick="setDir(0,-1)" style="font-size:16px;cursor:pointer;">&#9650;</button><div></div>
    <button onclick="setDir(-1,0)" style="font-size:16px;cursor:pointer;">&#9664;</button>
    <button onclick="startGame()" id="play-btn" style="font-size:11px;cursor:pointer;">Play</button>
    <button onclick="setDir(1,0)" style="font-size:16px;cursor:pointer;">&#9654;</button>
    <div></div>
    <button onclick="setDir(0,1)" style="font-size:16px;cursor:pointer;">&#9660;</button><div></div>
  </div>
</div>
<script>
const COLS=20,ROWS=20,CELL=18,GAP=1,SIZE=CELL+GAP;
const cv=document.getElementById('c'),ctx=cv.getContext('2d');
cv.width=COLS*SIZE-GAP; cv.height=ROWS*SIZE-GAP; cv.focus();

let snake,dir,nextDir,food,score,best=0,alive,timer;
const scoreEl=document.getElementById('score'),hiEl=document.getElementById('hi');
const msgEl=document.getElementById('msg'),subEl=document.getElementById('sub');

function reset(){
  snake=[{x:10,y:10},{x:9,y:10},{x:8,y:10}];
  dir={x:1,y:0}; nextDir={x:1,y:0};
  score=0; alive=true; scoreEl.textContent='0';
  placeFood(); draw();
}
function placeFood(){
  let p; do{ p={x:Math.floor(Math.random()*COLS),y:Math.floor(Math.random()*ROWS)};
  }while(snake.some(s=>s.x===p.x&&s.y===p.y)); food=p;
}
function rr(x,y,w,h,r){
  ctx.beginPath(); ctx.moveTo(x+r,y);
  ctx.lineTo(x+w-r,y); ctx.quadraticCurveTo(x+w,y,x+w,y+r);
  ctx.lineTo(x+w,y+h-r); ctx.quadraticCurveTo(x+w,y+h,x+w-r,y+h);
  ctx.lineTo(x+r,y+h); ctx.quadraticCurveTo(x,y+h,x,y+h-r);
  ctx.lineTo(x,y+r); ctx.quadraticCurveTo(x,y,x+r,y); ctx.fill();
}
function draw(){
  ctx.clearRect(0,0,cv.width,cv.height);
  for(let r=0;r<ROWS;r++) for(let c=0;c<COLS;c++){
    ctx.fillStyle='#f0f0ec'; ctx.fillRect(c*SIZE,r*SIZE,CELL,CELL);
  }
  ctx.fillStyle='#E24B4A'; rr(food.x*SIZE,food.y*SIZE,CELL,CELL,4);
  snake.forEach((s,i)=>{ ctx.fillStyle=i===0?'#0F6E56':'#1D9E75';
    rr(s.x*SIZE,s.y*SIZE,CELL,CELL,4); });
}
function tick(){
  if(!alive) return; dir=nextDir;
  const h={x:snake[0].x+dir.x,y:snake[0].y+dir.y};
  if(h.x<0||h.x>=COLS||h.y<0||h.y>=ROWS||snake.some(s=>s.x===h.x&&s.y===h.y)){
    alive=false; if(score>best){best=score;hiEl.textContent=best;}
    msgEl.textContent='Game over'; subEl.textContent='Press Play or Space';
    document.getElementById('play-btn').textContent='Play';
    clearInterval(timer); draw(); return;
  }
  snake.unshift(h);
  if(h.x===food.x&&h.y===food.y){score++;scoreEl.textContent=score;placeFood();}
  else snake.pop(); draw();
}
function setDir(dx,dy){ if(dir.x+dx===0&&dir.y+dy===0) return; nextDir={x:dx,y:dy}; }
function startGame(){
  clearInterval(timer); reset();
  msgEl.textContent=''; subEl.textContent='';
  document.getElementById('play-btn').textContent='Reset';
  timer=setInterval(tick,110);
}
document.addEventListener('keydown',e=>{
  if(e.key===' '||e.key==='Enter'){e.preventDefault();startGame();return;}
  const m={ArrowUp:[0,-1],ArrowDown:[0,1],ArrowLeft:[-1,0],ArrowRight:[1,0],
           w:[0,-1],s:[0,1],a:[-1,0],d:[1,0]};
  const d=m[e.key]; if(d){e.preventDefault();setDir(d[0],d[1]);}
});
msgEl.textContent='Snake'; subEl.textContent='Press Play or Space to start'; reset();
</script>
"""))